In [2]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class Document:
    content: str
    file_path: str
    file_type: str
    metadata: dict

In [3]:
document = Document(
    content="Hello RAG",
    file_path="example.txt",
    file_type=".txt",
    metadata={}
)

print(document)

Document(content='Hello RAG', file_path='example.txt', file_type='.txt', metadata={})


In [4]:
from pypdf import PdfReader
from docx import Document as DocxDocument


def load_pdf(file_path):
    reader = PdfReader(file_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        if text.strip():
            pages.append({
                "page_number": page_number,
                "content": text
            })

    return pages


def load_docx(file_path):
    doc = DocxDocument(file_path)

    paragraphs = []

    for paragraph in doc.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    return "\n".join(paragraphs)

In [5]:
def load_text(file_path):
    path = Path(file_path)

    return path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

In [6]:
def load_document(file_path):

    path = Path(file_path)

    extension = path.suffix.lower()

    if extension == ".pdf":

        pages = load_pdf(path)

        return Document(
            content="\n\n".join(
                page["content"]
                for page in pages
            ),
            file_path=str(path),
            file_type=extension,
            metadata={
                "pages": pages
            }
        )

    elif extension == ".docx":

        content = load_docx(path)

        return Document(
            content=content,
            file_path=str(path),
            file_type=extension,
            metadata={}
        )

    elif extension in [".txt", ".md"]:

        content = load_text(path)

        return Document(
            content=content,
            file_path=str(path),
            file_type=extension,
            metadata={}
        )

    else:

        raise ValueError(
            f"Unsupported file type: {extension}"
        )

In [7]:
from pathlib import Path

test_file = Path(
    "/workspace/data/documents/test.txt"
)

test_file.write_text(
    """Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.
""",
    encoding="utf-8"
)

print(test_file)

/workspace/data/documents/test.txt


In [8]:
document = load_document(test_file)

print("File:", document.file_path)
print("Type:", document.file_type)
print()
print(document.content)

File: /workspace/data/documents/test.txt
Type: .txt

Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.



In [9]:
from dataclasses import dataclass


@dataclass
class Chunk:
    content: str
    chunk_id: str
    file_path: str
    file_type: str
    metadata: dict

In [10]:
def chunk_text(
    text,
    chunk_size=1000,
    chunk_overlap=200
):
    if not text.strip():
        return []

    chunks = []

    start = 0
    text_length = len(text)

    while start < text_length:

        end = min(
            start + chunk_size,
            text_length
        )

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start = end - chunk_overlap

    return chunks

In [11]:
text = """
Retrieval-Augmented Generation combines
retrieval systems with language models.

The retrieval component searches a knowledge
base for relevant information.

The language model then uses the retrieved
information to generate an answer.

Chunking is important because documents are
usually too large to send directly to an LLM.
"""

chunks = chunk_text(
    text,
    chunk_size=150,
    chunk_overlap=30
)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- CHUNK {i} ---")
    print(chunk)

Number of chunks: 3

--- CHUNK 0 ---
Retrieval-Augmented Generation combines
retrieval systems with language models.

The retrieval component searches a knowledge
base for relevant infor

--- CHUNK 1 ---
wledge
base for relevant information.

The language model then uses the retrieved
information to generate an answer.

Chunking is important because do

--- CHUNK 2 ---
unking is important because documents are
usually too large to send directly to an LLM.


In [12]:
def create_chunks(document):
    
    text_chunks = chunk_text(
        document.content,
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = []

    for index, content in enumerate(text_chunks):

        chunk = Chunk(
            content=content,
            chunk_id=f"{Path(document.file_path).name}-{index}",
            file_path=document.file_path,
            file_type=document.file_type,
            metadata={
                **document.metadata,
                "chunk_index": index
            }
        )

        chunks.append(chunk)

    return chunks

In [13]:
chunks = create_chunks(document)

print("Total chunks:", len(chunks))

for chunk in chunks:
    print("\nID:", chunk.chunk_id)
    print(chunk.content)

Total chunks: 1

ID: test.txt-0
Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.


In [14]:
def create_pdf_chunks(document):

    chunks = []

    for page in document.metadata["pages"]:

        page_number = page["page_number"]
        page_text = page["content"]

        page_chunks = chunk_text(
            page_text,
            chunk_size=1000,
            chunk_overlap=200
        )

        for index, content in enumerate(page_chunks):

            chunk = Chunk(
                content=content,

                chunk_id=(
                    f"{Path(document.file_path).name}"
                    f"-page-{page_number}"
                    f"-chunk-{index}"
                ),

                file_path=document.file_path,

                file_type=document.file_type,

                metadata={
                    "page_number": page_number,
                    "chunk_index": index
                }
            )

            chunks.append(chunk)

    return chunks

In [15]:
def create_chunks(document):

    if document.file_type == ".pdf":
        return create_pdf_chunks(document)

    text_chunks = chunk_text(
        document.content,
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = []

    for index, content in enumerate(text_chunks):

        chunks.append(
            Chunk(
                content=content,

                chunk_id=(
                    f"{Path(document.file_path).name}"
                    f"-chunk-{index}"
                ),

                file_path=document.file_path,

                file_type=document.file_type,

                metadata={
                    **document.metadata,
                    "chunk_index": index
                }
            )
        )

    return chunks

In [16]:
chunks = create_chunks(document)

print("Total chunks:", len(chunks))

for chunk in chunks:
    print(
        chunk.chunk_id,
        "→",
        len(chunk.content),
        "characters"
    )

Total chunks: 1
test.txt-chunk-0 → 326 characters


In [17]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1085.61it/s]


Embedding model loaded


In [18]:
test_embedding = embedding_model.encode(
    chunks[0].content
)

print("Embedding type:", type(test_embedding))
print("Embedding dimensions:", len(test_embedding))

Embedding type: <class 'numpy.ndarray'>
Embedding dimensions: 384


In [19]:
def embed_chunks(chunks, embedding_model):

    texts = [
        chunk.content
        for chunk in chunks
    ]

    embeddings = embedding_model.encode(
        texts,
        show_progress_bar=True
    )

    return embeddings

In [24]:
embeddings = embed_chunks(
    chunks,
    embedding_model
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.97it/s]

Number of embeddings: 1
Embedding dimensions: 384


In [25]:
import chromadb
from pathlib import Path


CHROMA_PATH = "/workspace/data/chroma_data"

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

print("ChromaDB initialized")

ChromaDB initialized


In [26]:
collection = chroma_client.get_or_create_collection(
    name="documents"
)

print("Collection:", collection.name)

Collection: documents


In [27]:
def store_chunks(
    chunks,
    embeddings,
    collection
):

    ids = [
        chunk.chunk_id
        for chunk in chunks
    ]

    documents = [
        chunk.content
        for chunk in chunks
    ]

    metadatas = [
        {
            "file_path": chunk.file_path,
            "file_type": chunk.file_type,
            **chunk.metadata
        }
        for chunk in chunks
    ]

    collection.upsert(
        ids=ids,
        documents=documents,
        embeddings=embeddings.tolist(),
        metadatas=metadatas
    )

    print(
        f"Stored {len(chunks)} chunks"
    )

In [28]:
store_chunks(
    chunks,
    embeddings,
    collection
)

Stored 1 chunks


In [29]:
query = "What is retrieval augmented generation?"

query_embedding = embedding_model.encode(
    query
).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [30]:
for i, text in enumerate(
    results["documents"][0]
):

    print(f"\n--- RESULT {i + 1} ---")
    print(text)


--- RESULT 1 ---
Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.


In [31]:
print("Collection count:", collection.count())

Collection count: 1


In [32]:
query = "What is Retrieval-Augmented Generation?"

query_embedding = embedding_model.encode(
    query
).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

print(results)

{'ids': [['test.txt-chunk-0']], 'embeddings': None, 'documents': [['Retrieval-Augmented Generation (RAG) combines\ninformation retrieval with large language models.\n\nA RAG system first retrieves relevant information\nfrom a knowledge base and then provides that\ninformation to an LLM as context.\n\nThis can reduce hallucinations and allow an LLM\nto answer questions using private or external data.']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'chunk_index': 0, 'file_path': '/workspace/data/documents/test.txt', 'file_type': '.txt'}]], 'distances': [[0.6588929891586304]]}


In [33]:
for i in range(len(results["documents"][0])):

    print("=" * 60)
    print("RESULT:", i + 1)

    print("\nDistance:")
    print(results["distances"][0][i])

    print("\nMetadata:")
    print(results["metadatas"][0][i])

    print("\nText:")
    print(results["documents"][0][i])

RESULT: 1

Distance:
0.6588929891586304

Metadata:
{'chunk_index': 0, 'file_path': '/workspace/data/documents/test.txt', 'file_type': '.txt'}

Text:
Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.


In [34]:
def retrieve(query, collection, embedding_model, top_k=3):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    retrieved = []

    for i in range(len(results["documents"][0])):

        retrieved.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved

In [35]:
results = retrieve(
    "What is Retrieval-Augmented Generation?",
    collection,
    embedding_model,
    top_k=3
)

for result in results:

    print("=" * 60)
    print(result["text"])

Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.


In [36]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(results, start=1):

        metadata = result["metadata"]

        context_parts.append(
            f"""
SOURCE {i}

File: {metadata.get("file_name", "unknown")}
Page: {metadata.get("page_number", "unknown")}

Content:
{result["text"]}
"""
        )

    return "\n".join(context_parts)

In [37]:
context = build_context(results)

print(context)


SOURCE 1

File: unknown
Page: unknown

Content:
Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.



In [38]:
print(results[0]["metadata"])

{'chunk_index': 0, 'file_type': '.txt', 'file_path': '/workspace/data/documents/test.txt'}


In [39]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(results, start=1):

        metadata = result["metadata"]

        file_path = metadata.get(
            "file_path",
            "unknown"
        )

        chunk_index = metadata.get(
            "chunk_index",
            "unknown"
        )

        context_parts.append(
            f"""
SOURCE {i}

File: {file_path}
Chunk: {chunk_index}

Content:
{result["text"]}
"""
        )

    return "\n".join(context_parts)

In [40]:
context = build_context(results)

print(context)


SOURCE 1

File: /workspace/data/documents/test.txt
Chunk: 0

Content:
Retrieval-Augmented Generation (RAG) combines
information retrieval with large language models.

A RAG system first retrieves relevant information
from a knowledge base and then provides that
information to an LLM as context.

This can reduce hallucinations and allow an LLM
to answer questions using private or external data.



In [41]:
import os
from dotenv import load_dotenv

load_dotenv("/workspace/.env")

print("Groq key loaded:", bool(os.getenv("GROQ_API_KEY")))

Groq key loaded: True


In [42]:
from groq import Groq

groq_client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("Groq client ready")

Groq client ready


In [43]:
def generate_answer(query, context):

    prompt = f"""
You are a helpful document question-answering assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the answer cannot be found in the context, say:
"I couldn't find the answer in the provided documents."

Do not invent facts.

Context:
{context}

User question:
{query}

Answer:
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [44]:
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in one sentence."
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

Retrieval‑Augmented Generation (RAG) is a technique that combines a language model with a searchable external knowledge base, letting the model first retrieve relevant documents and then generate responses grounded in that retrieved information.


In [45]:
def ask_rag(query, top_k=3):

    results = retrieve(
        query=query,
        collection=collection,
        embedding_model=embedding_model,
        top_k=top_k
    )

    context = build_context(results)

    answer = generate_answer(
        query=query,
        context=context
    )

    return {
        "answer": answer,
        "sources": results,
        "context": context
    }

In [46]:
result = ask_rag(
    "What is Retrieval-Augmented Generation?"
)

print(result["answer"])

Retrieval‑Augmented Generation (RAG) is an approach that combines information retrieval with large language models. A RAG system first pulls relevant information from a knowledge base and then feeds that information to the LLM as context, which helps reduce hallucinations and lets the model answer questions using private or external data.


In [1]:
for source in result["sources"]:
    print("=" * 50)
    print(source["metadata"])
    print(source["text"])

NameError: name 'result' is not defined

In [1]:
results = retrieve(
    "What linguistic structures do the attention heads capture?",
    top_k=10
)

for i, r in enumerate(results, 1):
    print(f"SOURCE {i}")
    print(f"DISTANCE: {r['distance']}")
    print(f"FILE: {r['metadata'].get('file_name')}")
    print(f"TEXT: {r['text'][:300]}")
    print("-" * 80)

NameError: name 'retrieve' is not defined

In [2]:
from document_rag.rag import retrieve

ModuleNotFoundError: No module named 'document_rag'

In [1]:
import sys
sys.path.insert(0, "/workspace/src")

In [2]:
from document_rag.rag import retrieve

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4468.07it/s]


In [3]:
results = retrieve(
    "What linguistic structures do the attention heads capture?",
    top_k=10
)

for i, r in enumerate(results, 1):
    print(f"SOURCE {i}")
    print(f"DISTANCE: {r['distance']}")
    print(f"FILE: {r['metadata'].get('file_name')}")
    print(f"TEXT: {r['text'][:300]}")
    print("-" * 80)

SOURCE 1
DISTANCE: 0.9515706896781921
FILE: 1706.03762v7.pdf
TEXT: Attention Visualizations Input-Input Layer5 It is in this spirit that a majority of American governments have passed new laws since 2009 making the registration or voting process more difficult . It is in this spirit that a majority of American governments have passed new laws since 2009 making the 
--------------------------------------------------------------------------------
SOURCE 2
DISTANCE: 0.9519234895706177
FILE: 1706.03762v7.pdf
TEXT: Input-Input Layer5 The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . Figure 4: Two attention heads, also in layer 5 of 6
--------------------------------------------------------------------------------
SOURCE 3
DISTANCE: 1.002874732017517
FILE: 1706.03762v7.pdf
TEXT: Input-Input Layer5 The Law will neve

In [4]:
results = retrieve(
    "What linguistic structures do the attention heads capture?",
    top_k=30
)

for i, r in enumerate(results, 1):
    print(
        f"{i:02d} | "
        f"distance={r['distance']:.4f} | "
        f"page={r['metadata'].get('page_number')} | "
        f"{r['text'][:180]}"
    )

01 | distance=0.9516 | page=13 | Attention Visualizations Input-Input Layer5 It is in this spirit that a majority of American governments have passed new laws since 2009 making the registration or voting process m
02 | distance=0.9519 | page=14 | Input-Input Layer5 The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . The Law will never be perfect , but its applic
03 | distance=1.0029 | page=15 | Input-Input Layer5 The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . The Law will never be perfect , but its applic
04 | distance=1.0122 | page=12 | [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated corpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1
05 | distance=1.1144 | page=5 | output values. These are concatenated and once again projected, resulting in the final values, as depicted in Fi

In [5]:
query = "What linguistic structures do the attention heads capture?"

results = retrieve(
    query,
    top_k=30
)

keywords = [
    "linguistic",
    "syntactic",
    "semantic",
    "sentence structure",
    "attention heads",
    "attention head",
    "structure",
    "different tasks"
]

scored = []

for result in results:

    text = result["text"].lower()

    keyword_score = sum(
        1 for keyword in keywords
        if keyword in text
    )

    score = keyword_score - result["distance"]

    scored.append(
        (score, result)
    )

scored.sort(
    key=lambda x: x[0],
    reverse=True
)

for i, (score, r) in enumerate(scored[:10], 1):

    print(
        f"{i:02d} | "
        f"score={score:.3f} | "
        f"distance={r['distance']:.4f} | "
        f"page={r['metadata'].get('page_number')} | "
        f"{r['text'][:250]}"
    )

01 | score=4.805 | distance=1.1946 | page=7 | length n is smaller than the representation dimensionality d, which is most often the case with sentence representations used by state-of-the-art models in machine translations, such as word-piece [38] and byte-pair [31] representations. To improve c
02 | score=2.997 | distance=1.0029 | page=15 | Input-Input Layer5 The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . F
03 | score=1.048 | distance=0.9516 | page=13 | Attention Visualizations Input-Input Layer5 It is in this spirit that a majority of American governments have passed new laws since 2009 making the registration or voting process more difficult . It is in this spirit that a majority of American gover
04 | score=1.048 | distance=0.9519 | page=14 | Input-Input Layer5 The Law will never be perfect , but its a

In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

query = "What linguistic structures do the attention heads capture?"

results = retrieve(
    query,
    top_k=20
)

pairs = [
    [query, result["text"]]
    for result in results
]

scores = reranker.predict(pairs)

reranked = sorted(
    zip(scores, results),
    key=lambda x: x[0],
    reverse=True
)

for i, (score, r) in enumerate(reranked[:10], 1):

    print(
        f"{i:02d} | "
        f"rerank={score:.3f} | "
        f"distance={r['distance']:.4f} | "
        f"page={r['metadata'].get('page_number')} | "
        f"{r['text'][:250]}"
    )

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4814.92it/s]


01 | rerank=1.957 | distance=1.0029 | page=15 | Input-Input Layer5 The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . F
02 | rerank=-0.205 | distance=0.9516 | page=13 | Attention Visualizations Input-Input Layer5 It is in this spirit that a majority of American governments have passed new laws since 2009 making the registration or voting process more difficult . It is in this spirit that a majority of American gover
03 | rerank=-1.171 | distance=0.9519 | page=14 | Input-Input Layer5 The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . F
04 | rerank=-1.202 | distance=1.1946 | page=7 | length n is smaller than the representation dimension

In [8]:
from pathlib import Path

from document_rag.ingestion import load_pdf

pdf_path = Path(
    "/workspace/data/documents/1706.03762v7.pdf"
)

pages = load_pdf(pdf_path)

for page in pages:
    print(
        page.get("page_number"),
        len(page["text"]),
        page["text"][:100]
    )

1 2855 Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and
2 4251 1 Introduction Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural
3 1823 Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture us
4 2490 Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (ri
5 3181 output values. These are concatenated and once again projected, resulting in the final values, as de
6 3450 Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations for 
7 3300 length n is smaller than the representation dimensionality d, which is most often the case with sent
8 3178 Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the En
9 2973 Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the b
10 3111 Table 4: The Transfo